<a href="https://colab.research.google.com/github/towardsai/agentic-ai-engineering-course/blob/main/lessons/31_continuous_integration/notebook_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 31: Continuous Integration (CI) for AI Engineering

In this notebook, we'll practice the CI essentials covered in Lesson 31. You'll run formatting checks, linting, and tests to see how CI tools maintain code quality.

**Learning Objectives:**

- Understand Brown's CI configuration files
- Practice running formatting and linting checks with Ruff
- Learn to fix code quality issues automatically
- Run unit tests with mocked LLM responses

> **Exercise version.** This is the exercise notebook for Lesson 31. The full solution lives in [`notebook.ipynb`](https://colab.research.google.com/github/towardsai/agentic-ai-engineering-course/blob/main/lessons/31_continuous_integration/notebook.ipynb) in the same folder. Attempt each exercise before checking the solutions. No separate validation cells here: the Ruff and pytest commands in the solved cells are themselves the validators for what you write.

## 1. Setup


### Set Up Python Environment

**Google Colab:** Run the code cell below — it installs all required packages.

To set up your Python virtual environment using `uv` and load it into the Notebook, follow the step-by-step instructions from the `Course Admin` lesson from the beginning of the course.

**TL;DR:** Be sure the correct kernel pointing to your `uv` virtual environment is selected.


In [ ]:
# ============================================================
# Google Colab Setup — runs only when executed in Colab
# ============================================================
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import importlib
    import site
    import subprocess

    # Install the course package (published from pyproject.toml)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-U",
            "agentic-ai-engineering-course==1.0.0",
        ],
        check=True,
    )
    importlib.reload(site)  # make newly installed packages importable without restart

In [ ]:
if not IN_COLAB:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")

from utils import env

env.load(required_env_vars=["OPIK_API_KEY"])

In [ ]:
import os

if IN_COLAB:
    import writing_workflow

    %cd {list(writing_workflow.__path__)[0]}
else:
    %cd ../writing_workflow

## 2. Viewing Brown's CI Configuration

Let's examine Brown's actual CI configuration files to understand how CI is set up.

### 2.1 Pre-commit Configuration

The `.pre-commit-config.yaml` file defines Git hooks that run automatically before each commit. These hooks catch issues immediately in your local development environment.

Brown's pre-commit configuration includes three types of hooks:
1. **validate-pyproject** - Validates that `pyproject.toml` is structurally correct
2. **prettier** - Formats YAML and JSON configuration files consistently
3. **ruff-check** and **ruff-format** - Lints and formats Python code

In [ ]:
# Show the content of the file
!cat .pre-commit-config.yaml

### 2.2 Ruff Configuration

The `pyproject.toml` file contains Ruff's configuration in the `[tool.ruff]` section. This defines:
- **target-version**: Which Python version to target (py312 for Python 3.12)
- **line-length**: Maximum line length (140 characters for modern screens)
- **select rules**: Which linting rules to enable (F=Pyflakes, E=pycodestyle, I=isort)
- **known-first-party**: How to group imports correctly

In [ ]:
# Show the content of the file related to ruff
!grep -A 20 "\[tool.ruff\]" pyproject.toml

### 2.3 Makefile QA Targets

The Makefile provides convenient shortcuts for running CI commands. Instead of typing long `uv run ruff format --check src/ tests/ scripts/` commands, you can simply run `make format-check`.

The Makefile defines:
- **QA_FOLDERS** - Which directories to check (src/, tests/, scripts/)
- **format-check/format-fix** - Formatting commands
- **lint-check/lint-fix** - Linting commands
- **tests** - Test suite with the correct configuration
- **pre-commit** - Manual pre-commit hook execution

In [ ]:
# Show the commands in the Makefile related to QA
!sed -n '/# --- Tests & QA ---/,$p' Makefile | tail -n +2

## 3. Pre-commit Hooks (Local Enforcement)

Pre-commit hooks run automatically before each commit, catching issues before they enter version control.

> **If running from Colab:** the installed package isn't a git repository, but `pre-commit` requires one — we initialize it as a git repo first.

Let's run them manually:

In [ ]:
if IN_COLAB:
    !git init

!uv run pre-commit run --files ./**/*

These hooks will:
1. Validate your `pyproject.toml` structure
2. Format all YAML/JSON files with prettier
3. Lint Python code with ruff-check (and auto-fix issues)
4. Format Python code with ruff-format

If any hook fails, you'll see the error, fix it, re-stage the files, and commit again. This tight feedback loop keeps code quality high.


## 4. Running Formatting Checks

Now let's practice using Ruff's formatter. Instead of running it on Brown's existing code (which is already formatted), we'll create a simple Python file with formatting issues and fix them.

### 4.1 Create a Test File with Formatting Issues

### Exercise 1: Author a file that breaks the formatter

To understand what a formatter enforces, write code that violates it. You are about to do on purpose what CI exists to prevent.

**Learning goal:** Recognize the concrete style rules Ruff's formatter enforces by violating them deliberately.

**What you need to implement:**

Fill the heredoc with a short Python file that includes at least:

1. A function definition with extra spaces after `def`, parameters listed without spaces after the commas, and assignments with no spaces around operators
2. A list literal and a dict literal squeezed together without spaces after commas or colons
3. A class definition with extra spaces before its name and a cramped `__init__` in the same style

The code should be syntactically valid Python, badly formatted is not the same as broken.

**Key concepts:**

- Formatting violations are style-only: the interpreter accepts them, the formatter rejects them
- The heredoc (`<< 'EOF'`) writes your code verbatim into `test_formatting.py`

**Expected output:** `Created test_formatting.py`, then the check cell below reports the file would be reformatted, and after the fix cell you can diff what Ruff changed.

**Implementation hints:**

- Think of every space you would normally type and omit it
- Until you add real code, the check below reports the file as already formatted, that is your signal the exercise is not done

In [ ]:
%%bash
# === Exercise cell: fill in the gaps below ===

# Steps to complete: fill the heredoc with a short, syntactically valid Python
# file that deliberately violates formatting rules, following the briefing
# (cramped function, squeezed list/dict literals, cramped class)

cat > test_formatting.py << 'EOF'
# Your badly formatted Python goes here

EOF

# Smoke test
lines=$(grep -vc '^\s*#\|^\s*$' test_formatting.py || true)
if [ "$lines" -gt 3 ]; then echo "Created test_formatting.py"; else echo "(test_formatting.py has no real code yet, fill the heredoc above)"; fi

### 4.2 Check Formatting (Without Fixing)

Let's check if the file has formatting issues without modifying it:

In [ ]:
!uv run ruff format --check test_formatting.py

You'll see that Ruff reports the file would be reformatted. The `--check` flag means Ruff only reports issues without changing the file.

### 4.3 Auto-fix Formatting Issues

Now let's fix all the formatting issues automatically:

In [ ]:
!uv run ruff format test_formatting.py

Ruff will reformat the file to follow consistent style rules. Let's see the result:

In [ ]:
!cat test_formatting.py

Notice how Ruff has:
- Fixed spacing around operators (`x+y+z` → `x + y + z`)
- Added proper spacing in function signatures
- Formatted lists and dictionaries consistently
- Fixed class definition spacing

Let's remove the file now:

In [ ]:
!rm test_formatting.py

## 5. Running Linting Checks

Linting goes beyond formatting—it checks for bugs, code quality issues, and best practices. Let's create a file with linting issues and fix them.

### 5.1 Create a Test File with Linting Issues

### Exercise 2: Author a file that trips specific lint rules

Linting catches what formatting cannot: dead imports, duplicates, and outright bugs. Here you write a file that triggers three specific Ruff rules, then watch which ones auto-fix and which one needs a human.

**Learning goal:** Distinguish auto-fixable lint issues from logic errors by triggering both kinds.

**What you need to implement:**

Fill the heredoc with a short Python file that includes:

1. An import that is never used (triggers rule F401)
2. The same module imported a second time further down the file (triggers F811)
3. A call to a function name that does not exist anywhere (triggers F821)
4. A couple of small working functions around them, so the file looks like real code (use the other imports somewhere so only the intended issues fire)

**Key concepts:**

- F-prefixed rules come from Pyflakes: F401 unused import, F811 redefinition, F821 undefined name
- `ruff check --fix` repairs the first two mechanically, the undefined name is a logic error no tool should guess at

**Expected output:** the check cell below lists exactly your three rule codes, the fix cell removes two of them, and the final `cat` shows the undefined call still present.

**Implementation hints:**

- The rule descriptions in the markdown below the check cell double as your acceptance criteria
- Keep it minimal, three issues in ten lines beats ten issues you cannot trace

In [ ]:
%%bash
# === Exercise cell: fill in the gaps below ===

# Steps to complete: fill the heredoc with a short Python file that triggers
# exactly these lint rules, per the briefing:
#   - an unused import (F401)
#   - a duplicate import of the same module (F811)
#   - a call to an undefined function (F821)
# surrounded by a couple of small working functions

cat > test_linting.py << 'EOF'
# Your lint-rule-breaking Python goes here

EOF

# Smoke test
lines=$(grep -vc '^\s*#\|^\s*$' test_linting.py || true)
if [ "$lines" -gt 3 ]; then echo "Created test_linting.py"; else echo "(test_linting.py has no real code yet, fill the heredoc above)"; fi

### 5.2 Check Linting Issues (Without Fixing)

In [ ]:
!uv run ruff check test_linting.py

Ruff will report several issues:
- **F401**: Unused import (`json` is imported but never used)
- **F811**: Duplicate import (`sys` is imported twice)
- **F821**: Undefined name (`some_undefined_function` doesn't exist)

### 5.3 Auto-fix Linting Issues (Where Possible)

In [ ]:
!uv run ruff check --fix test_linting.py

Ruff will automatically fix:
- Remove unused imports
- Remove duplicate imports

But it won't fix the undefined name. That requires manual intervention since it's a logic error.


In [ ]:
!cat test_linting.py

Let's remove the file now:

In [ ]:
!rm test_linting.py

## 6. Running Unit Tests

Now let's run Brown's test suite with mocked LLM responses. The tests use fake models instead of real LLMs, making them fast, deterministic, and free.

### Exercise 3: Run the test suite the CI way

Brown's tests run against fake models, which is what makes them viable as a CI gate: fast, deterministic, and free. The one thing that makes that work is pointing the suite at the right config.

**Learning goal:** Run an AI application's test suite with mocked LLMs via configuration.

**What you need to implement:**

1. Write the command that runs Brown's full test suite verbosely through `uv run pytest`, with the `CONFIG_FILE` environment variable set inline to the debug config (the one under `configs/` that swaps every model for a fake)

**Key concepts:**

- An inline `VAR=value command` assignment scopes the environment variable to that single command
- The debug config is what severs the tests from real APIs: no keys, no cost, no nondeterminism

**Expected output:** verbose pytest output listing the domain, node, utility, and evaluation tests, all passing in under a minute with no API key errors.

**Implementation hints:**

- The two solved cells below run the same pattern on specific test folders, write yours before peeking
- If you see authentication or quota errors, the environment variable did not reach the command

In [ ]:
# === Exercise cell: fill in the gaps below ===

# Step to complete: run Brown's full test suite verbosely with uv run pytest,
# setting the CONFIG_FILE environment variable inline so every test uses the
# fake-model debug config from the configs folder

# Your command goes here:


print("(if you only see this line, add your test command above, pytest output will appear before it)")

The `-v` flag provides verbose output, showing each test as it runs. The `CONFIG_FILE=configs/debug.yaml` ensures all tests use fake models instead of real LLMs.

### 6.1 Running Specific Test Files

You can run tests for specific components:

In [ ]:
!CONFIG_FILE=configs/debug.yaml uv run pytest tests/brown/domain/ -v

In [ ]:
!CONFIG_FILE=configs/debug.yaml uv run pytest tests/brown/nodes/ -v

### 6.3 Understanding What's Being Tested

Brown's test suite includes:
- **Domain tests** (`tests/brown/domain/`): Testing Pydantic models and data structures without any LLM calls
- **Node tests** (`tests/brown/nodes/`): Testing agent nodes like ArticleWriter and ArticleReviewer with mocked LLM responses
- **Utility tests** (`tests/brown/utils/`): Testing helper functions
- **Evaluation tests** (`tests/brown/evals/`): Testing evaluation metrics and dataset handling

The complete test suite runs in under a minute and requires no API keys. Every test is deterministic.

## Stretch challenges

Want to go further? Try these on your own:

1. Re-create both of your broken files and run `uv run pre-commit run --files test_formatting.py test_linting.py`, watch the hooks catch and fix the same issues in one pass, then clean up.
2. Add `B` (flake8-bugbear) to the `select` list in a scratch copy of the Ruff config and run the check over `src/`, see whether Brown's real code trips any of the extra rules.
3. Write a file whose imports are out of order (standard library after third-party) and confirm that the `I` (isort) rules reorder them under `ruff check --fix`.